In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
import seaborn as sns
import pandas as pd
from collections import Counter
import cv2
from torchvision import models
import torch.nn.functional as F
import kagglehub
import shutil
from pathlib import Path
import json # Para salvar estatísticas

# Configuração de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo utilizado: {device}")

def download_and_prepare_dataset():
    """Baixa e prepara o dataset COVID-QU-Ex do Kaggle"""
    print("Baixando dataset COVID-QU-Ex...")
    
    # Download do dataset
    path = kagglehub.dataset_download("anasmohammedtahir/covidqu")
    print(f"Dataset baixado em: {path}")
    
    # Explorar estrutura do dataset (apenas para informação)
    print("\nExplorando estrutura do dataset (primeiros níveis)...")
    for root, dirs, files in os.walk(path):
        level = root.replace(str(path), '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:3]:  # Mostrar apenas os primeiros 3 arquivos
            print(f'{subindent}{file}')
        if len(files) > 3:
            print(f'{subindent}... and {len(files) - 3} more files')
        if level >= 2: # Limitar profundidade para visualização inicial
            del dirs[:] # Não descer mais
    
    return path

class COVIDQUDataset(Dataset):
    """Dataset personalizado para COVID-QU-Ex com pré-processamento avançado"""
    
    def __init__(self, dataset_path, split='train', transform=None, use_lung_masks=False):
        self.samples = []
        self.transform = transform
        self.use_lung_masks = use_lung_masks
        self.class_names = ['Normal', 'COVID-19', 'Non-COVID']
        
        # Mapeamento de labels
        self.label_mapping = {
            'Normal': 0,
            'COVID-19': 1, 
            'Non-COVID': 2
        }
        
        self.dataset_path = Path(dataset_path)
        self._load_samples(split)
        
        print(f"Dataset {split} carregado: {len(self.samples)} amostras")
        if len(self.samples) > 0:
            self._print_class_distribution()

    def _find_dataset_root(self):
        """Encontra a raiz mais provável contendo os dados de imagem."""
        possible_roots = [
            self.dataset_path / "Infection Segmentation Data" / "Infection Segmentation Data",
            self.dataset_path / "Lung Segmentation Data" / "Lung Segmentation Data",
            self.dataset_path / "Infection Segmentation Data",
            self.dataset_path / "Lung Segmentation Data",
            self.dataset_path
        ]
        
        for path in possible_roots:
            if path.exists():
                # Verifica se há subdiretórios de classes ou splits
                for item in path.iterdir():
                    if item.is_dir():
                        if any(c.lower() in item.name.lower() for c in ['normal', 'covid', 'pneumonia', 'non-covid', 'train', 'test', 'val']):
                            print(f"Raiz do dataset detectada: {path.relative_to(self.dataset_path) if path != self.dataset_path else 'base path'}")
                            return path
        print(f"Aviso: Não foi possível identificar uma raiz clara do dataset. Usando: {self.dataset_path}")
        return self.dataset_path

    def _load_samples(self, split):
        """Carrega amostras baseado no split especificado"""
        base_data_path = self._find_dataset_root()
        
        # Tentar carregar a partir de splits nomeados
        split_variants = {
            'train': ['Train', 'train', 'training'],
            'val': ['Val', 'val', 'validation', 'Validation'],
            'test': ['Test', 'test', 'testing']
        }
        
        found_split_path = None
        for variant in split_variants.get(split, [split]):
            # Tentar split_path diretamente
            potential_path = base_data_path / variant
            if potential_path.exists():
                found_split_path = potential_path
                break
            # Tentar split_path aninhado (ex: base_data_path/Train/Train)
            potential_nested_path = potential_path / variant
            if potential_nested_path.exists():
                found_split_path = potential_nested_path
                break
        
        if found_split_path and found_split_path.exists():
            print(f"Carregando dados para split '{split}' de: {found_split_path}")
            self._load_from_structured_path(found_split_path)
        else:
            print(f"Diretório para split '{split}' não encontrado diretamente em {base_data_path}. Tentando carregar todo o dataset e dividir manualmente.")
            self._load_all_and_split_manual(base_data_path, split)

    def _load_from_structured_path(self, path):
        """Carrega imagens de um caminho estruturado por classes."""
        for class_name in self.class_names:
            class_variants = self._get_class_variants(class_name)
            label = self.label_mapping[class_name]
            
            class_found = False
            for variant in class_variants:
                # Tentar diretamente o diretório da classe
                class_path = path / variant
                if class_path.exists():
                    self._add_images_from_directory(class_path, label)
                    class_found = True
                    break
                # Tentar subdiretório 'images' dentro da classe
                images_in_class_path = class_path / 'images'
                if images_in_class_path.exists():
                    self._add_images_from_directory(images_in_class_path, label)
                    class_found = True
                    break
            
            if not class_found:
                print(f"Aviso: Não encontrada estrutura clara para a classe '{class_name}' em {path}. Buscando recursivamente.")
                # Fallback: buscar recursivamente por imagens que contenham o nome da classe no path
                self._search_class_in_subdirectories(path, class_name)


    def _add_images_from_directory(self, directory_path, label):
        """Adiciona imagens de um diretório à lista de amostras."""
        image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']
        for ext in image_extensions:
            for img_file in directory_path.glob(ext):
                self.samples.append((str(img_file), label))
        if not self.samples: # Se não encontrou no primeiro nível, tentar recursivamente
            for ext in image_extensions:
                for img_file in directory_path.rglob(ext):
                    self.samples.append((str(img_file), label))

    def _get_class_variants(self, class_name):
        """Retorna variações possíveis do nome da classe"""
        variants = {
            'Normal': ['Normal', 'normal', 'NORMAL'],
            'COVID-19': ['COVID-19', 'covid', 'COVID', 'Covid-19', 'covid-19', 'COVID19'],
            'Non-COVID': ['Non-COVID', 'non-covid', 'NonCOVID', 'Pneumonia', 'pneumonia', 'Non-Covid', 'non-COVID']
        }
        return variants.get(class_name, [class_name])

    def _search_class_in_subdirectories(self, base_path, class_name):
        """Busca por imagens de uma classe em subdiretórios, sem estrutura predefinida."""
        label = self.label_mapping[class_name]
        class_variants = self._get_class_variants(class_name)
        
        for root, dirs, files in os.walk(base_path):
            root_path = Path(root)
            # Verifica se o diretório atual ou qualquer parte do caminho contém o nome da classe
            path_contains_class = any(variant.lower() in p.lower() for p in root_path.parts for variant in class_variants)
            
            if path_contains_class:
                for file in files:
                    if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                        img_path = root_path / file
                        # Evita adicionar duplicatas se já foi carregado por _load_from_structured_path
                        if (str(img_path), label) not in self.samples:
                            self.samples.append((str(img_path), label))
    
    def _load_all_and_split_manual(self, path, split):
        """Carrega todos os dados e faz split manual se não houver splits predefinidos."""
        print(f"Carregando todos os dados de {path} para split manual...")
        all_samples_raw = []
        
        for img_file in path.rglob('*.png'):
            class_name = self._determine_class_from_path(img_file)
            if class_name:
                label = self.label_mapping[class_name]
                all_samples_raw.append((str(img_file), label))
        
        for img_file in path.rglob('*.jpg'):
            class_name = self._determine_class_from_path(img_file)
            if class_name:
                label = self.label_mapping[class_name]
                all_samples_raw.append((str(img_file), label))
        
        # Filtrar amostras válidas (aquela onde a classe foi determinada)
        all_samples = [s for s in all_samples_raw if s[1] is not None]

        if not all_samples:
            print("Nenhuma imagem encontrada para o split manual. Verifique o caminho do dataset.")
            return

        print(f"Encontradas {len(all_samples)} imagens para split manual. Realizando split estratificado...")
        
        # Coleta os labels para split estratificado
        labels_only = [s[1] for s in all_samples]

        # Garantir que há dados suficientes para split
        if len(all_samples) < 3: # Não é possível fazer split 70/15/15
             print("Aviso: Poucas amostras para split estratificado ideal. Distribuindo todas as amostras para o conjunto de treinamento.")
             if split == 'train':
                 self.samples = all_samples
             else: # Val e Teste estarão vazios se não houver mais dados
                 self.samples = []
             return
        
        # Realizar split estratificado (70% treino, 15% validação, 15% teste)
        # Dividir em treino e temp (val + test)
        train_samples, temp_samples, _, temp_labels = train_test_split(
            all_samples, labels_only, test_size=0.3, random_state=42, stratify=labels_only
        )
        
        # Dividir temp em validação e teste (50/50 de temp, que é 15%/15% do total)
        val_samples, test_samples, _, _ = train_test_split(
            temp_samples, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
        )
        
        if split == 'train':
            self.samples = train_samples
        elif split == 'val':
            self.samples = val_samples
        elif split == 'test':
            self.samples = test_samples
            
        print(f"Split manual '{split}' concluído: {len(self.samples)} amostras.")
        
    def _determine_class_from_path(self, img_path):
        """Determina a classe baseada no caminho da imagem."""
        path_str = str(img_path).lower()
        
        if 'normal' in path_str:
            return 'Normal'
        elif 'covid' in path_str and 'non' not in path_str: # Evitar 'non-covid'
            return 'COVID-19'
        elif 'non-covid' in path_str or 'pneumonia' in path_str:
            return 'Non-COVID'
        
        # Tentar baseado no nome do arquivo (como fallback secundário)
        filename = img_path.name.lower()
        if 'normal' in filename:
            return 'Normal'
        elif 'covid' in filename and 'non' not in filename:
            return 'COVID-19'
        elif 'pneumonia' in filename or 'non' in filename: # 'non' no filename pode indicar non-covid
            return 'Non-COVID'
            
        # Se nenhuma classe for detectada, retornar None e a imagem será ignorada
        print(f"Aviso: Classe não determinada para {img_path}. Ignorando esta imagem.")
        return None

    def _print_class_distribution(self):
        """Imprime a distribuição das classes"""
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        print("Distribuição das classes:")
        total = len(self.samples)
        for class_idx, count in class_counts.items():
            class_name = self.class_names[class_idx]
            percentage = (count / total) * 100 if total > 0 else 0
            print(f"  {class_name}: {count} amostras ({percentage:.1f}%)")

    def get_class_weights(self):
        """Calcula pesos das classes para balanceamento"""
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        total_samples = len(self.samples)
        
        weights = []
        for i in range(len(self.class_names)):
            if i in class_counts and class_counts[i] > 0:
                # Peso inversamente proporcional à frequência, normalizado
                weight = total_samples / (len(self.class_names) * class_counts[i])
                weights.append(weight)
            else:
                weights.append(0.0) # Peso zero para classes não presentes no split atual
        
        # Normalizar os pesos para que somem 1 ou para um fator razoável
        sum_weights = sum(weights)
        if sum_weights > 0:
            weights = [w / sum_weights * len(self.class_names) for w in weights] # Escala para media 1
            
        return torch.FloatTensor(weights)

    def apply_lung_mask(self, image, img_path):
        """Aplica máscara pulmonar se disponível"""
        try:
            img_path_obj = Path(img_path)
            
            # Tentar diferentes localizações de máscara
            mask_paths = [
                # Mesmo diretório com sufixo _mask
                img_path_obj.parent / f"{img_path_obj.stem}_mask{img_path_obj.suffix}",
                # Diretório 'masks' ou 'lung masks' no mesmo nível do 'images'
                img_path_obj.parent.parent / "lung masks" / img_path_obj.name,
                img_path_obj.parent.parent / "masks" / img_path_obj.name,
                # Substituir 'images' por 'lung masks' ou 'masks' (caminhos absolutos)
                Path(str(img_path).replace(str(img_path_obj.parent.name), 'lung masks')).parent / img_path_obj.name,
                Path(str(img_path).replace(str(img_path_obj.parent.name), 'masks')).parent / img_path_obj.name,
                # Tentar em um diretório 'masks' ou 'lung masks' que esteja no mesmo nível dos diretórios de classe (e.g. 'Train/Normal' e 'Train/lung masks')
                img_path_obj.parents[2] / 'lung masks' / img_path_obj.name, # Assumindo estrutura como dataset_root/Split/Class/Image.png
                img_path_obj.parents[2] / 'masks' / img_path_obj.name,
            ]
            
            mask_path = None
            for potential_mask in mask_paths:
                if potential_mask.exists():
                    mask_path = potential_mask
                    break
            
            if mask_path:
                mask = Image.open(mask_path).convert('L')
                mask = mask.resize(image.size)
                
                image_array = np.array(image)
                mask_array = np.array(mask)
                
                # Normalizar máscara para 0-1
                mask_array = mask_array / 255.0
                
                # Aplicar máscara em cada canal
                if len(image_array.shape) == 3: # RGB
                    masked_image_array = np.zeros_like(image_array)
                    for c in range(image_array.shape[2]):
                        masked_image_array[:, :, c] = image_array[:, :, c] * mask_array
                else: # Grayscale (esperado para imagens médicas)
                    masked_image_array = image_array * mask_array
                
                return Image.fromarray(masked_image_array.astype(np.uint8))
                
        except Exception as e:
            print(f"Erro ao aplicar máscara em {img_path}: {e}")
            # print(traceback.format_exc()) # Para debug mais detalhado
        
        return image # Retorna a imagem original se a máscara não puder ser aplicada

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        try:
            image = Image.open(img_path).convert("RGB")
            
            # Aplicar máscara pulmonar se habilitada
            if self.use_lung_masks:
                image = self.apply_lung_mask(image, img_path)
            
            if self.transform:
                image = self.transform(image)
            else: # Garantir que a imagem é um tensor se nenhuma transformação é aplicada
                image = transforms.ToTensor()(image)
                image = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(image)

            return image, label
        except Exception as e:
            print(f"Erro ao carregar imagem {img_path}: {e}")
            # Retornar uma imagem preta em caso de erro, garantindo que seja um tensor
            dummy_image = torch.zeros(3, 224, 224) 
            return dummy_image, label


# Transformações avançadas com Data Augmentation específicas para CXR
def get_transforms(phase='train'):
    """Define transformações específicas para cada fase"""
    
    if phase == 'train':
        # Data Augmentation conservador para imagens médicas
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=5),  # Rotação menor para CXR
            transforms.ColorJitter(brightness=0.1, contrast=0.1),  # Ajustes menores
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])
    else:
        # Transformações para validação/teste
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])

class COVIDClassificationCNN(nn.Module):
    """CNN avançada para classificação COVID-19 com Transfer Learning"""
    
    def __init__(self, num_classes=3, pretrained=True, dropout_rate=0.4):
        super(COVIDClassificationCNN, self).__init__()
        
        # Backbone pré-treinado (ResNet50)
        self.backbone = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        
        # Substituir a última camada
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        # Classificador customizado otimizado para COVID-19
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )
        
        # Inicialização das camadas customizadas
        self._initialize_weights()

    def _initialize_weights(self):
        """Inicialização personalizada dos pesos"""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

class ModelTrainer:
    """Classe para gerenciar o treinamento do modelo"""
    
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, 
                 scheduler=None, device='cpu'):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        
        # Histórico de treinamento
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []
        
    def train_epoch(self):
        """Treina por uma época"""
        self.model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0
        
        for batch_idx, (images, labels) in enumerate(self.train_loader):
            images, labels = images.to(self.device), labels.to(self.device)
            
            # Forward pass
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            
            # Backward pass
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            # Estatísticas
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)
            
            # Log do progresso
            if (batch_idx + 1) % 100 == 0 or batch_idx == len(self.train_loader) - 1:
                print(f'  Batch {batch_idx+1}/{len(self.train_loader)}, Loss: {loss.item():.4f}')
        
        epoch_loss = running_loss / len(self.train_loader)
        epoch_acc = correct_predictions / total_samples
        
        return epoch_loss, epoch_acc
    
    def validate_epoch(self):
        """Valida por uma época"""
        self.model.eval()
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0
        
        with torch.no_grad():
            for images, labels in self.val_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                correct_predictions += (predicted == labels).sum().item()
                total_samples += labels.size(0)
        
        epoch_loss = running_loss / len(self.val_loader)
        epoch_acc = correct_predictions / total_samples
        
        return epoch_loss, epoch_acc
    
    def train(self, num_epochs, early_stopping_patience=15):
        """Treinamento completo com early stopping"""
        best_val_acc = 0.0
        patience_counter = 0
        
        print(f"Iniciando treinamento por {num_epochs} épocas...")
        print("-" * 60)
        
        for epoch in range(num_epochs):
            print(f'Época {epoch+1}/{num_epochs}')
            
            # Treinamento
            train_loss, train_acc = self.train_epoch()
            
            # Validação
            val_loss, val_acc = self.validate_epoch()
            
            # Atualizar scheduler
            if self.scheduler:
                old_lr = self.optimizer.param_groups[0]['lr']
                self.scheduler.step(val_loss)
                new_lr = self.optimizer.param_groups[0]['lr']
                if new_lr != old_lr:
                    print(f'Learning rate reduzido de {old_lr:.6f} para {new_lr:.6f}')
            
            # Salvar histórico
            self.train_losses.append(train_loss)
            self.train_accuracies.append(train_acc)
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_acc)
            
            print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
            print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
            
            # Early stopping
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                # Salvar melhor modelo
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_acc': val_acc,
                    'val_loss': val_loss
                }, 'best_covid_model.pth')
                print(f'Novo melhor modelo salvo! Val Acc: {val_acc:.4f}')
            else:
                patience_counter += 1
            
            if patience_counter >= early_stopping_patience:
                print(f'Early stopping após {early_stopping_patience} épocas sem melhoria')
                break
            
            print("-" * 60)
        
        print(f'Treinamento concluído! Melhor Val Acc: {best_val_acc:.4f}')
        return best_val_acc
    
    def plot_training_history(self):
        """Plota o histórico de treinamento"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss
        ax1.plot(self.train_losses, label='Train Loss', color='blue')
        ax1.plot(self.val_losses, label='Validation Loss', color='red')
        ax1.set_title('Curva de Loss - COVID-19 Classification')
        ax1.set_xlabel('Época')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True)
        
        # Accuracy
        ax2.plot(self.train_accuracies, label='Train Accuracy', color='blue')
        ax2.plot(self.val_accuracies, label='Validation Accuracy', color='red')
        ax2.set_title('Curva de Acurácia - COVID-19 Classification')
        ax2.set_xlabel('Época')
        ax2.set_ylabel('Acurácia')
        ax2.legend()
        ax2.grid(True)
        
        plt.tight_layout()
        plt.savefig('covid_training_history.png', dpi=300, bbox_inches='tight')
        plt.show()

def evaluate_model(model, test_loader, device, class_names):
    """Avaliação completa do modelo"""
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probabilities = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    # Métricas
    print("=== RELATÓRIO DE CLASSIFICAÇÃO COVID-19 ===")
    print(classification_report(all_labels, all_predictions, 
                                target_names=class_names, digits=4))
    
    # Matriz de confusão
    cm = confusion_matrix(all_labels, all_predictions)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Matriz de Confusão - COVID-19 Classification')
    plt.ylabel('Rótulo Verdadeiro')
    plt.xlabel('Rótulo Predito')
    plt.savefig('covid_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Calcular AUC para classificação multiclasse
    all_probabilities = np.array(all_probabilities)
    try:
        auc_scores = {}
        for i, class_name in enumerate(class_names):
            binary_labels = (np.array(all_labels) == i).astype(int)
            # roc_auc_score precisa de pelo menos duas classes presentes
            if len(np.unique(binary_labels)) > 1:
                auc = roc_auc_score(binary_labels, all_probabilities[:, i])
                auc_scores[class_name] = auc
                print(f"AUC {class_name}: {auc:.4f}")
            else:
                print(f"Não foi possível calcular AUC para {class_name} (apenas uma classe presente no teste).")
        
        if auc_scores: # Calcular média apenas se houver AUCs válidos
            mean_auc = np.mean(list(auc_scores.values()))
            print(f"AUC Médio: {mean_auc:.4f}")
        else:
            print("Nenhum AUC válido calculado.")
            
        # Métricas específicas para COVID-19
        if 'COVID-19' in class_names:
            covid_report = classification_report(all_labels, all_predictions, 
                                                 target_names=class_names, 
                                                 output_dict=True)['COVID-19']
            covid_precision = covid_report['precision']
            covid_recall = covid_report['recall']
            covid_f1 = covid_report['f1-score']
            
            print(f"\n=== MÉTRICAS ESPECÍFICAS COVID-19 ===")
            print(f"Precisão COVID-19: {covid_precision:.4f}")
            print(f"Recall COVID-19: {covid_recall:.4f}")
            print(f"F1-Score COVID-19: {covid_f1:.4f}")
        else:
            print("\nClasse 'COVID-19' não encontrada nas predições de teste.")
        
    except Exception as e:
        print(f"Erro ao calcular AUC ou métricas específicas: {e}")
    
    return all_predictions, all_labels, all_probabilities

def visualize_predictions(model, test_loader, device, class_names, num_samples=16):
    """Visualiza predições em um conjunto de amostras"""
    model.eval()
    
    # Coletar amostras
    images_to_show = []
    labels_to_show = []
    predictions_to_show = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            if len(images_to_show) >= num_samples:
                break
                
            images_cpu = images.cpu() # Manter no CPU para desnormalização
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            # Desnormalizar imagens para visualização
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            
            for i in range(min(images.size(0), num_samples - len(images_to_show))):
                img = images_cpu[i] * std + mean # Use images_cpu aqui
                img = torch.clamp(img, 0, 1)
                
                images_to_show.append(img.permute(1, 2, 0).numpy())
                labels_to_show.append(labels[i].item())
                predictions_to_show.append(predicted[i].item())
    
    # Plotar amostras
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    axes = axes.ravel()
    
    for i in range(min(len(images_to_show), 16)):
        ax = axes[i]
        ax.imshow(images_to_show[i], cmap='gray')
        
        true_label = class_names[labels_to_show[i]]
        pred_label = class_names[predictions_to_show[i]]
        
        color = 'green' if labels_to_show[i] == predictions_to_show[i] else 'red'
        ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color, fontsize=10)
        ax.axis('off')
    
    # Ocultar eixos vazios
    for i in range(len(images_to_show), 16):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig('covid_predictions_visualization.png', dpi=300, bbox_inches='tight')
    plt.show()

def analyze_misclassifications(predictions, labels, class_names):
    """Analisa erros de classificação"""
    print("\n=== ANÁLISE DE ERROS DE CLASSIFICAÇÃO ===")
    
    # Matriz de confusão detalhada
    cm = confusion_matrix(labels, predictions)
    
    print("\nErros mais comuns:")
    for i, true_class in enumerate(class_names):
        for j, pred_class in enumerate(class_names):
            if i != j and cm[i][j] > 0:
                # Evitar divisão por zero se a linha for toda zero (classe sem amostras verdadeiras)
                total_true_class_samples = np.sum(cm[i])
                error_rate = cm[i][j] / total_true_class_samples if total_true_class_samples > 0 else 0
                print(f"{true_class} classificado como {pred_class}: "
                      f"{cm[i][j]} casos ({error_rate:.2%})")
    
    # Análise específica para COVID-19
    if 'COVID-19' in class_names:
        covid_idx = class_names.index('COVID-19')
        covid_true_positives = cm[covid_idx][covid_idx]
        total_covid_actual = np.sum(cm[covid_idx])
        total_covid_predicted = np.sum(cm[:, covid_idx])

        covid_false_negatives = total_covid_actual - covid_true_positives
        covid_false_positives = total_covid_predicted - covid_true_positives
        
        print(f"\n=== ANÁLISE ESPECÍFICA COVID-19 ===")
        print(f"Verdadeiros Positivos (COVID-19 corretamente identificado): {covid_true_positives}")
        print(f"Falsos Negativos (COVID-19 não identificado): {covid_false_negatives}")
        print(f"Falsos Positivos (Não-COVID/Normal identificado como COVID-19): {covid_false_positives}")
        
        if total_covid_actual > 0:
            print(f"Taxa de Falsos Negativos (COVID-19 não detectado): {covid_false_negatives / total_covid_actual:.2%}")
        if total_covid_predicted > 0:
            print(f"Taxa de Falsos Positivos (alarme falso COVID-19): {covid_false_positives / total_covid_predicted:.2%}")

        # Detalhe dos Falsos Positivos de COVID-19
        print("Detalhe dos Falsos Positivos (Predito como COVID-19, mas não é):")
        for i, true_class in enumerate(class_names):
            if i != covid_idx and cm[i][covid_idx] > 0:
                print(f"  - {true_class} classificado como COVID-19: {cm[i][covid_idx]} casos")

    else:
        print("Classe 'COVID-19' não encontrada para análise específica.")

def create_covid_report(model, test_loader, device, class_names, dataset_stats):
    """Cria relatório detalhado do modelo COVID-19"""
    print("\n" + "="*80)
    print("RELATÓRIO DETALHADO - CLASSIFICADOR COVID-19 CNN")
    print("Dataset: COVID-QU-Ex")
    print("="*80)
    
    # Informações do dataset
    print(f"\nINFORMAÇÕES DO DATASET:")
    print(f"Amostras de treino utilizadas: {dataset_stats.get('train_samples', 'N/A')}")
    print(f"Amostras de validação: {dataset_stats.get('val_samples', 'N/A')}")
    print(f"Amostras de teste: {dataset_stats.get('test_samples', 'N/A')}")
    
    # Avaliação detalhada
    predictions, labels, probabilities = evaluate_model(model, test_loader, device, class_names)
    
    # Análise de erros
    analyze_misclassifications(predictions, labels, class_names)
    
    # Visualizar predições
    print("\nVisualizando algumas predições do conjunto de teste...")
    visualize_predictions(model, test_loader, device, class_names)
    
    # Métricas por classe
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score
    precision, recall, f1, support = precision_recall_fscore_support(labels, predictions, average=None, labels=list(range(len(class_names))))
    accuracy = accuracy_score(labels, predictions)

    print(f"\n=== MÉTRICAS DETALHADAS POR CLASSE ===")
    print(f"Acurácia Geral: {accuracy:.4f}")
    for i, class_name in enumerate(class_names):
        print(f"\n{class_name.upper()}:")
        print(f"  Precisão: {precision[i]:.4f}")
        print(f"  Recall: {recall[i]:.4f}")
        print(f"  F1-Score: {f1[i]:.4f}")
        print(f"  Suporte: {support[i]} amostras")
    
    # Salvar relatório em arquivo
    with open('covid_classification_report.txt', 'w', encoding='utf-8') as f:
        f.write("RELATÓRIO DETALHADO - CLASSIFICADOR COVID-19 CNN\n")
        f.write("Dataset: COVID-QU-Ex\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Acurácia Geral: {accuracy:.4f}\n\n")
        
        report = classification_report(labels, predictions, target_names=class_names, digits=4)
        f.write("RELATÓRIO DE CLASSIFICAÇÃO:\n")
        f.write(report)
        f.write("\n")
        
        cm = confusion_matrix(labels, predictions)
        f.write("MATRIZ DE CONFUSÃO:\n")
        f.write("Formato: [Normal, COVID-19, Non-COVID]\n")
        for i, row in enumerate(cm):
            f.write(f"{class_names[i]}: {row}\n")

    print(f"\nRelatório salvo em: covid_classification_report.txt")
    return predictions, labels, probabilities

def save_model_for_deployment(model, transform_params, class_names, model_info):
    """Salva modelo para deployment"""
    deployment_package = {
        'model_state_dict': model.state_dict(),
        'model_architecture': 'ResNet50-based COVID Classifier',
        'class_names': class_names,
        'transform_params': {
            'mean': [0.485, 0.456, 0.406],
            'std': [0.229, 0.224, 0.225],
            'size': (224, 224)
        },
        'model_info': model_info,
        'dataset': 'COVID-QU-Ex',
        'num_classes': len(class_names)
    }
    
    torch.save(deployment_package, 'covid_classifier_deployment.pth')
    print("Modelo salvo para deployment: covid_classifier_deployment.pth")

def main():
    """Função principal atualizada"""
    
    # Configurações otimizadas para COVID-QU-Ex
    BATCH_SIZE = 32
    NUM_EPOCHS = 50
    LEARNING_RATE = 0.0001
    WEIGHT_DECAY = 1e-4
    USE_LUNG_MASKS = False # Ativar se quiser usar máscaras pulmonares
    
    print("=== CNN AVANÇADA PARA CLASSIFICAÇÃO COVID-19 ===")
    print("Dataset: COVID-QU-Ex (Kaggle)")
    print("Classes: Normal, COVID-19, Non-COVID")
    print("=" * 60)
    
    try:
        # Baixar dataset
        dataset_base_path = download_and_prepare_dataset()
        
        # Carregar datasets
        print("\n" + "=" * 60)
        print("CARREGANDO DATASETS...")
        print("=" * 60)
        
        train_dataset = COVIDQUDataset(
            dataset_base_path, 
            split='train',
            transform=get_transforms('train'),
            use_lung_masks=USE_LUNG_MASKS
        )
        
        val_dataset = COVIDQUDataset(
            dataset_base_path, 
            split='val',
            transform=get_transforms('val'),
            use_lung_masks=USE_LUNG_MASKS
        )
        
        test_dataset = COVIDQUDataset(
            dataset_base_path, 
            split='test', 
            transform=get_transforms('test'),
            use_lung_masks=USE_LUNG_MASKS
        )
        
        # Verificar se os datasets foram carregados com sucesso
        if len(train_dataset) == 0:
            print("❌ ERRO CRÍTICO: Nenhuma amostra de treinamento encontrada!")
            print("Não é possível prosseguir sem dados de treinamento. Verifique a estrutura do dataset.")
            return
        if len(val_dataset) == 0:
            print("❌ ERRO: Nenhuma amostra de validação encontrada! O modelo pode estar sobreajustando sem validação adequada.")
        if len(test_dataset) == 0:
            print("❌ ERRO: Nenhuma amostra de teste encontrada! A avaliação final será comprometida.")

        # Calcular pesos das classes (apenas para o conjunto de treino)
        class_weights = train_dataset.get_class_weights().to(device)
        print(f"\nPesos das classes (treino): {class_weights}")
        
        # DataLoaders
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                                  shuffle=True, num_workers=os.cpu_count() // 2 or 1, pin_memory=True) # Ajustar num_workers
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
                                shuffle=False, num_workers=os.cpu_count() // 2 or 1, pin_memory=True)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, 
                                 shuffle=False, num_workers=os.cpu_count() // 2 or 1, pin_memory=True)
        
        # Modelo
        print("\nInicializando modelo COVID-19...")
        model = COVIDClassificationCNN(num_classes=3, pretrained=True, dropout_rate=0.4)
        model = model.to(device)
        
        # Critério com pesos das classes
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        
        # Otimizador
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, 
                                weight_decay=WEIGHT_DECAY)
        
        # Scheduler
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5 #verbose=True
        )
        
        # Treinador
        trainer = ModelTrainer(model, train_loader, val_loader, criterion, 
                               optimizer, scheduler, device)
        
        # Treinamento
        best_val_acc = trainer.train(NUM_EPOCHS, early_stopping_patience=15)
        
        # Plotar histórico
        trainer.plot_training_history()
        
        # Carregar melhor modelo para avaliação
        if os.path.exists('best_covid_model.pth'):
            print("\nCarregando melhor modelo para avaliação final...")
            checkpoint = torch.load('best_covid_model.pth', map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            print("\n⚠️  Nenhum 'best_covid_model.pth' encontrado. Usando o modelo treinado mais recente para avaliação.")
            
        # Coletar estatísticas do dataset para o relatório
        dataset_stats = {
            'train_samples': len(train_dataset),
            'val_samples': len(val_dataset),
            'test_samples': len(test_dataset),
            'class_weights': class_weights.cpu().numpy().tolist()
        }

        # Avaliação final e relatório
        create_covid_report(model, test_loader, device, train_dataset.class_names, dataset_stats)
        
        # Salvar modelo para deployment
        model_info = {
            'best_val_accuracy': best_val_acc,
            'epochs_trained': len(trainer.train_losses),
            'final_train_loss': trainer.train_losses[-1] if trainer.train_losses else 'N/A',
            'final_val_loss': trainer.val_losses[-1] if trainer.val_losses else 'N/A',
            'learning_rate': LEARNING_RATE,
            'batch_size': BATCH_SIZE,
            'use_lung_masks': USE_LUNG_MASKS
        }
        save_model_for_deployment(model, get_transforms('val'), train_dataset.class_names, model_info)
        
        print("\nTreinamento e avaliação concluídos com sucesso!")
        
    except Exception as e:
        print(f"Erro durante a execução: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Dispositivo utilizado: cuda
SISTEMA DE CLASSIFICAÇÃO COVID-19 COM RAIOS-X
Baixando dataset COVID-QU-Ex...
Dataset baixado em: /home/jose/.cache/kagglehub/datasets/anasmohammedtahir/covidqu/versions/7

Explorando estrutura completa do dataset...
7/ (1 arquivos)
  Infection Segmentation Data/ (0 arquivos)
    Infection Segmentation Data/ (0 arquivos)
      Val/ (0 arquivos)
        Normal/ (0 arquivos)
        Non-COVID/ (0 arquivos)
        COVID-19/ (0 arquivos)
      Test/ (0 arquivos)
        Normal/ (0 arquivos)
        Non-COVID/ (0 arquivos)
        COVID-19/ (0 arquivos)
      Train/ (0 arquivos)
        Normal/ (0 arquivos)
        Non-COVID/ (0 arquivos)
        COVID-19/ (0 arquivos)
  Lung Segmentation Data/ (0 arquivos)
    Lung Segmentation Data/ (0 arquivos)
      Val/ (0 arquivos)
        Normal/ (0 arquivos)
        Non-COVID/ (0 arquivos)
        COVID-19/ (0 arquivos)
      Test/ (0 arquivos)
        Normal/ (0 arquivos)
        Non-COVID/ (0 arquivos)
        COVID-19

Traceback (most recent call last):
  File "/tmp/ipykernel_69529/1701087324.py", line 951, in main
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'
